This is a public presentation copy: code and final explanations are retained; learning drafts remain in the original local notebook. Saved outputs and local metadata are cleared. Run from the notebooks directory after installing the project and obtaining the source CSV. See ../reports/model_card.md for the final experiment results and ../docs/bank_transfer_design.md for the separate hypothetical bank requirements. Ablation and interaction ideas in EDA are proposed checks, not completed experiments.


# Stage 2: Business-Question-Driven EDA

Original working title: `探索相关关系`

Dataset: IBM Telco Customer Churn public fictional dataset.

Purpose: Explore associations between customer characteristics and the recorded
`Churn` label. Findings are descriptive and do not establish causality.

In [ ]:
from pathlib import Path

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import pandas as pd

In [ ]:
import seaborn as sns

In [ ]:
data_path = Path("../data/raw/customer_churn.csv")

In [ ]:
assert data_path.exists(),f"dataset not found: {data_path.resolve()}"

In [ ]:
df_raw = pd.read_csv(data_path)

In [ ]:
df = df_raw.copy()

In [ ]:
print("raw shape",df_raw.shape)

In [ ]:
print("analysis shape",df.shape)

In [ ]:
assert df.shape == (7043,21)

In [ ]:
assert df["customerID"].is_unique

In [ ]:
assert set(df["Churn"].unique()) == {"No","Yes"}

In [ ]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors = "coerce",
)

In [ ]:
tcm = df["TotalCharges"].isna()

In [ ]:
print("missing value", tcm.sum())

In [ ]:
df.loc[tcm,
    ["customerID", "tenure", "MonthlyCharges", "TotalCharges"],
    ]

In [ ]:
assert tcm.sum() == 11
assert df.loc[tcm,"tenure"].eq(0).all()

In [ ]:
df["ChurnFlag"] = df["Churn"].map({
    "No": 0,
    "Yes": 1,
})

In [ ]:
assert df["ChurnFlag"].notna().all()
assert set(df["ChurnFlag"].unique()) == {0,1}

In [ ]:
print(df["ChurnFlag"].mean())

In [ ]:
print("raw dtype", df_raw["TotalCharges"].dtype)

In [ ]:
print("analysis dtype:", df["TotalCharges"].dtype)

In [ ]:
c_columns = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
]

In [ ]:
df[c_columns].describe().T

In [ ]:
ca_columns = [
    "Contract",
    "PaymentMethod",
    "InternetService",
    "TechSupport",
]

In [ ]:
for column in ca_columns:
    ca_summary = pd.DataFrame({
        "customer_count": df[column].value_counts
        (dropna = False),
        "customer_share": df[column].value_counts(
            normalize=True,
            dropna=False,
        ),
    })
    print(f"\n{column}")
    display(ca_summary)


In [ ]:
for column in ca_columns:
    plt.figure(figsize=(8,4))

    sns.countplot(
        data=df,
        x=column,
        order=df[column].value_counts().index,
    )
    plt.title(f"Customer count by {column}")
    plt.xlabel(column)
    plt.ylabel("customer count")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()
    

In [ ]:
con_churn = (df.groupby("Contract",dropna=False)
            .agg(
                customer_count=("customerID","size"),
                churn_count=("ChurnFlag","sum"),
                churn_rate=("ChurnFlag","mean")
            )
             .sort_values("churn_rate",ascending=False)
            )

In [ ]:
con_churn

In [ ]:
assert con_churn["customer_count"].sum() == len(df)
assert con_churn["churn_count"].sum() == df["ChurnFlag"].sum()
assert con_churn["churn_rate"].between(0,1).all()

In [ ]:
plot_data = con_churn.reset_index()

plt.figure(figsize=(8, 4))
sns.barplot(
    data=plot_data,
    x="Contract",
    y="churn_rate",
)

plt.axhline(
    df["ChurnFlag"].mean(),
    color="red",
    linestyle="--",
    label="Overall churn rate",
)

plt.title("Observed churn rate by Contract")
plt.xlabel("Contract")
plt.ylabel("Churn rate")
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
cachurntable= {}

In [ ]:
for column in ["InternetService", "TechSupport", "PaymentMethod"]:
    summary = (
        df.groupby(column,dropna=False)
        .agg(
            customer_count=("customerID","size"),
            churn_count=("ChurnFlag","sum"),
            churn_rate=("ChurnFlag","mean"),
        )
        .sort_values("churn_rate",ascending=False)
    )
    assert summary["customer_count"].sum() == len(df)
    assert summary["churn_count"].sum() == df["ChurnFlag"].sum()
    assert summary["churn_rate"].between(0,1).all()
    cachurntable[column] = summary
    display(summary)

    plot_data = summary.reset_index()

    plt.figure(figsize=(8, 4))
    sns.barplot(data=plot_data, x=column, y="churn_rate")
    plt.axhline(
        df["ChurnFlag"].mean(),
        color="red",
        linestyle="--",
        label="Overall churn rate",
        )
    plt.title(f"Observed churn rate by {column}")
    plt.xlabel(column)
    plt.ylabel("Churn rate")
    plt.ylim(0, 1)
    plt.xticks(rotation=20, ha="right")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
continuous_churn_summary = (
    df.groupby("Churn")[["tenure", "MonthlyCharges", "TotalCharges"]]
    .agg(["count", "mean", "median"])
)

continuous_churn_summary

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(data=df, x="Churn", y="tenure")
plt.title("Tenure by recorded churn status")
plt.xlabel("Recorded churn")
plt.ylabel("Tenure (months)")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(data=df, x="Churn", y="MonthlyCharges")
plt.title("Monthly charges by recorded churn status")
plt.xlabel("Recorded churn")
plt.ylabel("Monthly charges")
plt.tight_layout()
plt.show()

In [ ]:
tenure_bins = [-1, 6, 12, 24, 48, 72]
tenure_labels = ["0-6", "7-12", "13-24", "25-48", "49-72"]
df["TenureBand"] =pd.cut(
    df["tenure"],
    bins = tenure_bins,
    labels= tenure_labels,
)

In [ ]:
tenure_churn = (
    df.groupby("TenureBand",observed=True)
    .agg(
        customer_count=("customerID","size"),
        churn_count=("ChurnFlag","sum"),
        churn_rate=("ChurnFlag","mean"),
    )
)

In [ ]:
tenure_churn

In [ ]:
assert tenure_churn["customer_count"].sum() == len(df)
assert tenure_churn["churn_count"].sum() == df["ChurnFlag"].sum()
assert tenure_churn["churn_rate"].between(0,1).all()


## 建模假设 1：Tenure 与流失

### 我的原始假设

> Customers who just open a new account for a few months get high churn rate.

我后来又把它扩展为一个组合假设：

> Customers with high MonthlyCharges, short tenure and low TotalCharges would be more likely to churn.

原始业务直觉是：**刚开始使用服务、客户关系时长较短的新客户可能具有更高的流失风险。**

### 规范化的可验证假设

**假设：** `tenure` 有助于预测客户是否流失，tenure 较短的客户可能具有更高的观察流失风险。

**EDA 证据：** 流失客户的 tenure 中位数为 **10个月**，未流失客户为 **38个月**；0–6个月客户共有1,481人，观察流失率为 **52.94%**，而49–72个月客户的观察流失率为 **9.51%**。

**后续验证：** 在完全相同的训练集、验证集和交叉验证切分下，比较包含与不包含 `tenure` 的模型表现。只有当加入 `tenure` 后验证集指标稳定改善，才能认为它具有增量预测价值。

**潜在风险：** `tenure` 与 `Contract`、`TotalCharges` 等变量存在较强关系，因此单变量 EDA 中观察到的差异可能受到其他变量混杂。当前结果只能说明相关性，不能证明较短 tenure 直接导致流失。

**组合关系的后续探索：** 可以进一步检验“较短 tenure 与较高 MonthlyCharges 同时出现时，流失风险是否更高”。该关系可通过分组交叉表、交互项或树模型验证，不能仅凭两个单变量结果直接下结论。

## 建模假设 2：Payment Method 与流失


**Hypothesis：** `PaymentMethod` 可能提供额外的流失预测信息，其中使用 Electronic check 的客户预计具有更高的预测流失风险。

**EDA evidence：** Electronic check 客户的观察流失率为 **45.29%（1,071/2,365）**，高于 Bank transfer (automatic) 的 **16.71%（258/1,544）**和 Credit card (automatic) 的 **15.24%（232/1,522）**。

**Later validation：** 在完全相同的训练集、验证集和交叉验证切分下，比较包含与不包含 `PaymentMethod` 的模型表现。若加入该变量后验证指标稳定改善，再认为它具有增量预测价值；同时检查 Logistic Regression 系数或最终模型解释。

**Risk or limitation：** 支付方式不是随机分配的，它可能与 `Contract`、`tenure`、`PaperlessBilling` 及客户的服务使用习惯相关，因此观察到的高流失率可能受到这些变量混杂。当前结果只能说明相关性，不能证明 Electronic check 直接导致客户流失。

## Formal Split of My Original ASSUMPTION3


### Hypothesis 3A — Fiber-optic service

**Hypothesis:** `InternetService` 可能提供额外的流失预测信息，其中 Fiber-optic 客户预计具有更高的预测流失风险。

**EDA evidence:** Fiber-optic 客户的观察流失率为 **41.89%（1,297/3,096）**，高于 DSL 客户的 **18.96%（459/2,421）**和没有互联网服务客户的 **7.40%（113/1,526）**。

**Later validation:** 在完全相同的交叉验证切分下，比较包含和不包含 `InternetService` 的模型表现，并检查 Logistic Regression 系数或最终模型解释。

**Risk or limitation:** Fiber-optic 与 MonthlyCharges、Contract、tenure 和其他服务选择可能相关。当前差异属于描述性关联，不能证明 Fiber optic 直接导致流失。

### Hypothesis 3B — TechSupport conditional on InternetService

**Hypothesis:** 在拥有互联网服务的客户中，`TechSupport = No` 可能提供额外的流失预测信息，并与更高的预测流失风险相关。

**EDA evidence:** `TechSupport = No` 客户的观察流失率为 **41.64%（1,446/3,473）**，而 `TechSupport = Yes` 客户为 **15.17%（310/2,044）**。数据将没有互联网服务的客户单独记录为 `No internet service`，没有混入普通的 `No` 类别。

**Later validation:** 在相同交叉验证切分下比较包含和不包含 `TechSupport` 的模型表现；如需检验服务类型是否改变 TechSupport 的关系，可进一步比较只有主效应的模型与加入 `InternetService × TechSupport` 交互项的模型。

**Risk or limitation:** `InternetService` 决定客户是否可能拥有 TechSupport，因此两者存在结构性依赖。若把 `No` 与 `No internet service` 合并，会把“没有购买支持”和“根本没有互联网服务”混为一谈，并产生混杂或编码偏差。此外，Contract、tenure 与 MonthlyCharges 也可能影响观察到的差异。

In [ ]:
continuous_columns = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
]

df[continuous_columns].describe().T

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(16,4))
for column ,axis in zip(continuous_columns,axes):
    sns.histplot(data=df,x=column,kde=True,ax=axis)
    axis.set_title(f"Distribution of {column}")
    axis.set_ylabel("Customer count")
plt.tight_layout()
plt.show()

In [ ]:
# Integrity checks for the Codex-assisted numerical summary.
assert len(df) == 7043
assert int(df["ChurnFlag"].sum()) == 1869
assert round(df["ChurnFlag"].mean(), 4) == 0.2654

assert int(con_churn.loc["Month-to-month", "customer_count"]) == 3875
assert int(con_churn.loc["Month-to-month", "churn_count"]) == 1655
assert round(con_churn.loc["Month-to-month", "churn_rate"], 4) == 0.4271
assert int(con_churn.loc["Two year", "customer_count"]) == 1695
assert int(con_churn.loc["Two year", "churn_count"]) == 48
assert round(con_churn.loc["Two year", "churn_rate"], 4) == 0.0283

internet_summary = cachurntable["InternetService"]
assert int(internet_summary.loc["Fiber optic", "customer_count"]) == 3096
assert int(internet_summary.loc["Fiber optic", "churn_count"]) == 1297
assert round(internet_summary.loc["Fiber optic", "churn_rate"], 4) == 0.4189
assert round(internet_summary.loc["DSL", "churn_rate"], 4) == 0.1896
assert round(internet_summary.loc["No", "churn_rate"], 4) == 0.0740

support_summary = cachurntable["TechSupport"]
assert int(support_summary.loc["No", "customer_count"]) == 3473
assert int(support_summary.loc["No", "churn_count"]) == 1446
assert round(support_summary.loc["No", "churn_rate"], 4) == 0.4164
assert round(support_summary.loc["Yes", "churn_rate"], 4) == 0.1517

payment_summary = cachurntable["PaymentMethod"]
assert int(payment_summary.loc["Electronic check", "customer_count"]) == 2365
assert int(payment_summary.loc["Electronic check", "churn_count"]) == 1071
assert round(payment_summary.loc["Electronic check", "churn_rate"], 4) == 0.4529
assert round(payment_summary.loc["Bank transfer (automatic)", "churn_rate"], 4) == 0.1671
assert round(payment_summary.loc["Credit card (automatic)", "churn_rate"], 4) == 0.1524

assert int(tenure_churn.loc["0-6", "customer_count"]) == 1481
assert int(tenure_churn.loc["0-6", "churn_count"]) == 784
assert round(tenure_churn.loc["0-6", "churn_rate"], 4) == 0.5294
assert int(tenure_churn.loc["49-72", "customer_count"]) == 2239
assert int(tenure_churn.loc["49-72", "churn_count"]) == 213
assert round(tenure_churn.loc["49-72", "churn_rate"], 4) == 0.0951

assert continuous_churn_summary.loc["Yes", ("tenure", "median")] == 10
assert continuous_churn_summary.loc["No", ("tenure", "median")] == 38
assert round(
    continuous_churn_summary.loc["Yes", ("MonthlyCharges", "mean")], 2
) == 74.44
assert round(
    continuous_churn_summary.loc["No", ("MonthlyCharges", "mean")], 2
) == 61.27
assert round(
    continuous_churn_summary.loc["Yes", ("TotalCharges", "median")], 2
) == 703.55
assert round(
    continuous_churn_summary.loc["No", ("TotalCharges", "median")], 2
) == 1683.60


## 数字结论
1. **整体流失率：** 7,043 名客户中有 1,869 名记录为流失，流失率为 **26.54%**；反过来说，多数类“不流失”约占 **73.46%**。因此后续模型不能只看 accuracy，还要与永远预测 `No` 的基线比较，并重点观察召回率等指标。

2. **合同类型：** Month-to-month 客户的流失率为 **42.71%（1,655/3,875）**，Two-year 客户为 **2.83%（48/1,695）**。短期合同与较高流失明显相关，但 Contract 同时与 tenure、费用和服务选择有关，不能据此认定合同类型直接导致流失。

3. **互联网服务：** Fiber-optic 客户的流失率为 **41.89%（1,297/3,096）**，DSL 为 **18.96%（459/2,421）**，没有互联网服务的客户约为 **7.40%（113/1,526）**。该差异可能与价格、服务体验或客户结构有关，后续应与其他特征共同验证。

4. **技术支持：** 没有技术支持的客户流失率为 **41.64%（1,446/3,473）**，有技术支持的客户为 **15.17%（310/2,044）**。TechSupport 可能提供预测信息，但客户是否选择支持服务本身也可能受到合同和服务类型影响。

5. **付款方式：** Electronic-check 客户的流失率为 **45.29%（1,071/2,365）**，自动银行转账为 **16.71%（258/1,544）**，自动信用卡付款为 **15.24%（232/1,522）**。付款方式可能区分出不同客户群，但不能解释为电子支票本身造成流失。

6. **客户关系时长：** 流失客户的 tenure 中位数为 **10个月**，未流失客户为 **38个月**；0–6个月客户流失率为 **52.94%（784/1,481）**，49–72个月客户为 **9.51%（213/2,239）**。新客户阶段可能是重要风险窗口，但 tenure 与 Contract、TotalCharges 存在较强关系。

7. **费用结构：** 流失客户的 MonthlyCharges 均值约为 **74.44**，未流失客户约为 **61.27**；流失客户的 TotalCharges 中位数反而较低，为 **703.55**，未流失客户为 **1,683.60**。这种组合与流失客户 tenure 较短相符，也说明 TotalCharges 不能脱离 tenure 单独解释。

In [ ]:
df["RiskCombo"] = (
    df["InternetService"].eq("Fiber optic")
    & df["OnlineSecurity"].eq("No")
    & df["TechSupport"].eq("No")
)

combo_summary = (
    df.groupby("RiskCombo")
    .agg(
        customer_count=("customerID", "size"),
        churn_count=("ChurnFlag", "sum"),
        churn_rate=("ChurnFlag", "mean"),
    )
)

combo_summary